In [ ]:
# ============================================
# PHASE 1 — UNSW-NB15 — Model Training & Evaluation
# ============================================

import pandas as pd
import numpy as np
import time
import os
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, matthews_corrcoef, confusion_matrix)
from scipy.stats import wilcoxon
from itertools import combinations

# ---- 1. LOAD PROCESSED DATA ----
train_df = pd.read_csv(r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\UNSW-NB15\processed\UNSWNB15_train_cleaned.csv")

# Drop attack_cat to prevent data leakage (it directly encodes the label)
X_train = train_df.drop(columns=['label', 'attack_cat'])
y_train = train_df['label']

# ---- 2. CLASS LABELS ----
POS_LABEL = 1
LABELS = [0, 1]

# ---- 3. MODELS ----
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'SVM': SVC(random_state=42)
}

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
print("Cross-validation set up: 10 folds")

# ---- 4. FOLD LOOP ----
results = {name: {'accuracy': [], 'precision': [], 'recall': [], 'f1': [], 'fpr': [], 'mcc': []} for name in models}

X_train_array = X_train.values
y_train_array = y_train.values

for fold_num, (train_idx, val_idx) in enumerate(skf.split(X_train_array, y_train_array), 1):
    X_fold_train, X_fold_val = X_train_array[train_idx], X_train_array[val_idx]
    y_fold_train, y_fold_val = y_train_array[train_idx], y_train_array[val_idx]

    print(f"\n--- Fold {fold_num}/10 ---")

    for name, model in models.items():
        start = time.time()
        model.fit(X_fold_train, y_fold_train)
        y_pred = model.predict(X_fold_val)
        elapsed = time.time() - start

        acc = accuracy_score(y_fold_val, y_pred)
        prec = precision_score(y_fold_val, y_pred, pos_label=POS_LABEL)
        rec = recall_score(y_fold_val, y_pred, pos_label=POS_LABEL)
        f1 = f1_score(y_fold_val, y_pred, pos_label=POS_LABEL)
        mcc = matthews_corrcoef(y_fold_val, y_pred)

        tn, fp, fn, tp = confusion_matrix(y_fold_val, y_pred, labels=LABELS).ravel()
        fpr = fp / (fp + tn)

        results[name]['accuracy'].append(acc)
        results[name]['precision'].append(prec)
        results[name]['recall'].append(rec)
        results[name]['f1'].append(f1)
        results[name]['fpr'].append(fpr)
        results[name]['mcc'].append(mcc)

        print(f"{name}: acc={acc:.4f}, f1={f1:.4f}, mcc={mcc:.4f} ({elapsed:.1f}s)")

print("\nAll folds complete!")

# ---- 5. SUMMARY TABLE ----
summary = pd.DataFrame({
    name: {metric: sum(vals)/len(vals) for metric, vals in results[name].items()}
    for name in results
}).T
summary = summary.round(4)
print(summary)

# ---- 6. WILCOXON SIGNED-RANK TESTS ----
model_names = list(results.keys())
print("\nWilcoxon signed-rank test results (F1 score, pairwise):\n")

for model_a, model_b in combinations(model_names, 2):
    scores_a = results[model_a]['f1']
    scores_b = results[model_b]['f1']
    stat, p_value = wilcoxon(scores_a, scores_b)
    significant = "SIGNIFICANT (p < 0.05)" if p_value < 0.05 else "not significant"
    print(f"{model_a} vs {model_b}: p={p_value:.4f} -> {significant}")

# ---- 7. SAVE RESULTS ----
results_folder = r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\UNSW-NB15\results"
os.makedirs(results_folder, exist_ok=True)

summary.to_csv(os.path.join(results_folder, "Phase1_UNSWNB15_summary.csv"))

fold_results = pd.DataFrame({
    (name, metric): vals
    for name in results
    for metric, vals in results[name].items()
})
fold_results.to_csv(os.path.join(results_folder, "Phase1_UNSWNB15_foldlevel.csv"))

print("Results saved successfully!")
print("Summary:", os.path.join(results_folder, "Phase1_UNSWNB15_summary.csv"))
print("Fold-level:", os.path.join(results_folder, "Phase1_UNSWNB15_foldlevel.csv"))

Cross-validation set up: 10 folds

--- Fold 1/10 ---
Logistic Regression: acc=0.9041, f1=0.9139, mcc=0.8168 (1.4s)
KNN: acc=0.9050, f1=0.9098, mcc=0.8100 (3.5s)
Decision Tree: acc=0.9158, f1=0.9190, mcc=0.8314 (3.3s)
